# ABSA Pipeline — LoRA M1 + Binary Sentiment M2
## Architecture
```
final.csv (2400+ labeled)  ──► Iterative Stratified Split
                                    ├── Train (~70%)
                                    ├── Val   (~15%)  ← early stopping / threshold tuning
                                    └── Test  (~15%)  ← LOCKED, open once

final_nonlabel.csv (1.6M)  ──► Filter English + Remove overlap ──► Unlabeled Pool

STEP 1 — DAPT
  roberta-base + MLM on (Train sentences + Unlabeled Pool) → Domain-Adapted RoBERTa

STEP 2 — Multi-task Model 1  [LoRA rank=32]
  Domain-Adapted RoBERTa + LoRA
    ├── Head 1: Related/Unrelated     (all sentences,  BCE, weight 0.2)
    ├── Head 2: General/Technical     (related only,   BCE, weight 0.2)
    └── Head 3: Aspect (N-class MLB)  (technical only, BCE, weight 0.6)

STEP 3 — Sentiment Model 2  [Binary Neg/Pos] (due to the low number of Neutral)
  yangheng/deberta-v3-base-absa-v1.1
```

In [1]:
!pip install scikit-multilearn -q
!pip install langdetect deep-translator -q
!pip install nlpaug -q
!pip install peft -q
!pip install -U torchao peft transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.4/89.4 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 15.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.5/410.5 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 147.2 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from datasets import Dataset as HFDataset

from transformers import (
    AutoTokenizer, AutoModel,
    AutoModelForSequenceClassification,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    TrainingArguments, Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    get_linear_schedule_with_warmup,
)

from peft import LoraConfig, get_peft_model

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import train_test_split
from skmultilearn.model_selection import iterative_train_test_split

import nlpaug.augmenter.word as naw
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0
import uuid
import warnings, os, json, re

warnings.filterwarnings('ignore')
from sklearn.metrics import f1_score, fbeta_score

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

Device: cuda
GPU: Tesla T4


In [3]:
class CFG:
    # ── Paths ──
    LABELED_PATH    = 'final.csv'
    UNLABELED_PATH  = 'final_nonlabel.csv'
    TEXT_COL        = 'sentence_for_model'
    ASPECT_COL      = 'target_aspect'
    SENTIMENT_COL   = 'target_sentiment'
    IS_RELATED_COL  = 'is_related'
    IS_TECH_COL     = 'is_technical'
    ROW_ID_COL      = 'row_id'

    # ── Models ──
    BACKBONE        = 'roberta-base'
    SENTIMENT_MODEL = 'yangheng/deberta-v3-base-absa-v1.1'
    DAPT_OUTPUT     = './dapt_roberta'
    M1_OUTPUT       = './model1_multitask'
    M2_OUTPUT       = './model2_sentiment'

    # ── Split ──
    SEED            = 42
    VAL_RATIO       = 0.15
    TEST_RATIO      = 0.15

    # ── DAPT ──
    DAPT_MAX_LEN    = 128
    DAPT_BATCH      = 32
    DAPT_EPOCHS     = 4
    DAPT_LR         = 5e-5
    DAPT_UNLABELED_SAMPLE = 300_000

    # ── Model 1 ──
    M1_MAX_LEN      = 128
    M1_BATCH        = 16
    M1_EPOCHS       = 50
    M1_LR           = 5e-5
    M1_WEIGHT_DECAY = 0.05
    M1_WARMUP_RATIO = 0.1
    LOSS_W_H1       = 0.2
    LOSS_W_H2       = 0.2
    LOSS_W_H3       = 0.6
    USE_UNCERTAINTY_WEIGHTS = False
    M1_PATIENCE     = 4
    M1_MONITOR_METRIC = 'f1_mean'   # 'f1_mean' | 'f1_h3' | 'f1_h1' ...
    FREEZE_H1_AFTER_EPOCH = 10
    AUG_RARE_THRESHOLD    = 150
    AUG_NUM_PER_SAMPLE    = 2

    # ── LORA ──
    M1_LORA_R       = 32
    M1_LORA_ALPHA   = 32 #
    M1_LORA_DROPOUT = 0.1
    M1_LORA_TARGET  = ['query', 'key', 'value', 'dense']

    # ── Model 2 ──
    M2_MAX_LEN      = 128
    M2_BATCH        = 30
    M2_EPOCHS       = 10
    M2_LR           = 2e-5
    M2_PATIENCE     = 3
    FORCE_RETRAIN_M2 = False  # set True to skip checkpoint and train again for Model 2

    # ── Inference ──
    UNSURE_LABEL    = 'unsure'
    M2_INFERENCE_BATCH = 64

torch.manual_seed(CFG.SEED)
np.random.seed(CFG.SEED)
print('Config loaded. hehehe')

Config loaded. hehehe


## 2. Data Loading & Preparation
Load labeled data, assign unique UUIDs to rows, and extract valid aspect-sentiment pairs while excluding non-target classes.

Load labeled data and assign unique UUIDs to rows.

In [4]:
df = pd.read_csv(CFG.LABELED_PATH).dropna(subset=[CFG.TEXT_COL])
print(f'Labeled rows: {len(df)}')
print('Columns:', df.columns.tolist())

# row_id UUID
df[CFG.ROW_ID_COL] = [str(uuid.uuid4()) for _ in range(len(df))]
df = df.reset_index(drop=True)
print(f'row_id sample: {df[CFG.ROW_ID_COL].iloc[0]}')
print('row_id unique check:', df[CFG.ROW_ID_COL].is_unique)  # Need to be True

Labeled rows: 2410
Columns: ['original_id', 'parent_asin', 'product_title', 'rating', 'review_context', 'sentence', 'sentence_for_model', 'target_aspect', 'target_sentiment', 'is_related', 'is_technical']
row_id sample: 6c0c7052-200f-4270-be88-06c99615f343
row_id unique check: True


Parse & encode aspects

In [5]:
EXCLUDE_ASPECTS = {'general', 'unrelated'}

def parse_and_validate_pairs(aspect_str, sentiment_str):
    """
    Parse Aspect and Sentiment.
    Drop short rows -> consistency in M1 and M2 data
    """
    if pd.isna(aspect_str) or pd.isna(sentiment_str):
        return [], []

    raw_aspects = [a.strip().lower() for a in str(aspect_str).split(';')]
    raw_sents   = [s.strip() for s in str(sentiment_str).split(';')]

    if len(raw_aspects) != len(raw_sents):
        return [], []

    valid_aspects, valid_sents = [], []
    for asp, sent in zip(raw_aspects, raw_sents):
        if asp and asp not in EXCLUDE_ASPECTS:
            try:
                sent_int = int(float(sent))
                valid_aspects.append(asp)
                valid_sents.append(sent_int)
            except ValueError:
                pass

    return valid_aspects, valid_sents

Apply Parser

In [6]:
parsed_results = df.apply(
    lambda x: parse_and_validate_pairs(x[CFG.ASPECT_COL], x[CFG.SENTIMENT_COL]),
    axis=1
)
df['aspects_parsed'] = parsed_results.apply(lambda x: x[0])
df['sentiments_parsed'] = parsed_results.apply(lambda x: x[1])
df['is_valid_pairs'] = df['aspects_parsed'].apply(len) > 0

## 3. Iterative Stratified Split (multilabel-safe)
Perform iterative stratified splitting on aspect-sentiment combinations to create balanced train/val/test sets, fit the binarizer without leakage, and merge non-technical rows.

Only valid, related, and technical rows are extracted

In [7]:
df_tech = df[
    (df[CFG.IS_RELATED_COL] == 1) &
    (df[CFG.IS_TECH_COL] == 1) &
    (df['is_valid_pairs'] == True)
].copy().reset_index(drop=True)

Create combo labels aspect_sentiment for iterative stratified split

In [8]:
def get_combo_labels(aspects, sentiments):
    return [f"{a}_{s}" for a, s in zip(aspects, sentiments)]

In [9]:
df_tech['combo_labels'] = df_tech.apply(
    lambda x: get_combo_labels(x['aspects_parsed'], x['sentiments_parsed']),
    axis=1
)

Convert combo labels into multi-hot binary

In [10]:
mlb_combo = MultiLabelBinarizer()
y_combo = np.array(mlb_combo.fit_transform(df_tech['combo_labels']), dtype=float)
X_tech_ids = df_tech[[CFG.ROW_ID_COL]].values

In [11]:
X_trainval_id, y_trainval_combo, X_test_id, y_test_combo = iterative_train_test_split(
    X_tech_ids, y_combo, test_size=CFG.TEST_RATIO
)
val_ratio_adjusted = CFG.VAL_RATIO / (1 - CFG.TEST_RATIO)
X_train_id, y_train_combo, X_val_id, y_val_combo = iterative_train_test_split(
    X_trainval_id, y_trainval_combo, test_size=val_ratio_adjusted
)

Reconstruct DataFrames from split IDs and fit the MultiLabelBinarizer strictly on the training set to define aspect classes without data leakage.

In [12]:
def get_df_from_ids(X_id_array):
    ids = [x[0] for x in X_id_array]
    return df_tech[df_tech[CFG.ROW_ID_COL].isin(ids)].copy()

In [13]:
train_df_raw = get_df_from_ids(X_train_id)
val_df_raw   = get_df_from_ids(X_val_id)
test_df_raw  = get_df_from_ids(X_test_id)

mlb = MultiLabelBinarizer()
mlb.fit(train_df_raw['aspects_parsed'].tolist())

ASPECT_CLASSES = list(mlb.classes_)
N_ASPECTS      = len(ASPECT_CLASSES)

print(f'\nAspect classes ({N_ASPECTS}): {ASPECT_CLASSES}')
train_aspects_list = train_df_raw['aspects_parsed'].tolist()
for cls in ASPECT_CLASSES:
    cnt = sum(cls in a for a in train_aspects_list)
    print(f'  {cls:<20}: {cnt} train samples ({cnt/len(train_df_raw)*100:.1f}%)')


Aspect classes (7): ['battery', 'design & build', 'display & audio', 'input', 'performance', 'price', 'software']
  battery             : 54 train samples (6.7%)
  design & build      : 245 train samples (30.2%)
  display & audio     : 104 train samples (12.8%)
  input               : 72 train samples (8.9%)
  performance         : 216 train samples (26.6%)
  price               : 109 train samples (13.4%)
  software            : 111 train samples (13.7%)


Transform parsed aspects into multi-hot arrays and perform set-intersection assertions

In [14]:
def apply_aspect_labels(df_raw):
    d      = df_raw.copy()
    y_m1   = mlb.transform(d['aspects_parsed'].tolist())
    d['labels'] = list(y_m1)
    return d

In [15]:
train_df = apply_aspect_labels(train_df_raw)
val_df   = apply_aspect_labels(val_df_raw)
test_df  = apply_aspect_labels(test_df_raw)

y_train = np.array(train_df['labels'].tolist(), dtype=float)
y_val   = np.array(val_df['labels'].tolist(),   dtype=float)
y_test  = np.array(test_df['labels'].tolist(),  dtype=float)

train_df = train_df.drop(columns=['labels'])
val_df   = val_df.drop(columns=['labels'])
test_df  = test_df.drop(columns=['labels'])

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test (LOCKED): {len(test_df)}')
print(f'y_train shape: {y_train.shape} | y_val: {y_val.shape} | y_test: {y_test.shape}')
print('\nLabel distribution per aspect (train):')
for i, cls in enumerate(ASPECT_CLASSES):
    print(f'  {cls:<20}: {int(y_train[:,i].sum())} positives')

Train: 812 | Val: 175 | Test (LOCKED): 180
y_train shape: (812, 7) | y_val: (175, 7) | y_test: (180, 7)

Label distribution per aspect (train):
  battery             : 54 positives
  design & build      : 245 positives
  display & audio     : 104 positives
  input               : 72 positives
  performance         : 216 positives
  price               : 109 positives
  software            : 111 positives


In [16]:
train_ids_set = set(train_df[CFG.ROW_ID_COL])
val_ids_set   = set(val_df[CFG.ROW_ID_COL])
test_ids_set  = set(test_df[CFG.ROW_ID_COL])

# Protection
assert len(train_ids_set & val_ids_set) == 0,  "LEAK: train and val NOT EMPTY!"
assert len(train_ids_set & test_ids_set) == 0, "LEAK: train and test NOT EMPTY!"
assert len(val_ids_set & test_ids_set) == 0,   "LEAK: val and test NOT EMPTY!"
print("Split integrity check: OK! NO LEAK.")

df_full_train = train_df.copy()
df_full_val   = val_df.copy()
df_full_test  = test_df.copy()

Split integrity check: OK! NO LEAK.


Stratify-split non-technical rows, concatenate them with technical sets, and assert mutual exclusivity

In [17]:
df_non_tech = df[
    ~df[CFG.ROW_ID_COL].isin(df_tech[CFG.ROW_ID_COL])
].copy()

non_tech_train, temp = train_test_split(df_non_tech, test_size=CFG.TEST_RATIO + CFG.VAL_RATIO,
                                         stratify=df_non_tech[CFG.IS_RELATED_COL], random_state=CFG.SEED)
non_tech_val, non_tech_test = train_test_split(temp, test_size=0.5,
                                                stratify=temp[CFG.IS_RELATED_COL], random_state=CFG.SEED)

df_full_train = pd.concat([df_full_train, non_tech_train]).reset_index(drop=True)
df_full_val   = pd.concat([df_full_val,   non_tech_val]).reset_index(drop=True)
df_full_test  = pd.concat([df_full_test,  non_tech_test]).reset_index(drop=True)

In [18]:
full_train_ids = set(df_full_train[CFG.ROW_ID_COL])
full_val_ids   = set(df_full_val[CFG.ROW_ID_COL])
full_test_ids  = set(df_full_test[CFG.ROW_ID_COL])

assert len(full_train_ids & full_val_ids)  == 0, "LEAK: full train ∩ val NOT EMPTY!"
assert len(full_train_ids & full_test_ids) == 0, "LEAK: full train ∩ test NOT EMPTY!"
assert len(full_val_ids   & full_test_ids) == 0, "LEAK: full val ∩ test NOT EMPTY!"
print("Split integrity check (full): OK! NO LEAK.")

print(f'\nFull splits (all rows):')
print(f'  Train: {len(df_full_train)} | Val: {len(df_full_val)} | Test: {len(df_full_test)}')

Split integrity check (full): OK! NO LEAK.

Full splits (all rows):
  Train: 1682 | Val: 361 | Test: 367


## 7. Model 2 — Sentiment (Fine-tuned, yangheng/deberta-absa)
Approach: Fine-tune on per-aspect sentiment triples from final.csv

Example: target_aspect = "battery;camera" → target_sentiment = "0;1".

Dataset construction:
build_m2_triples() extracts (sentence, aspect, sentiment_label) triples from each split, correctly handling both single-aspect and multi-aspect rows. Length-mismatch rows are dropped entirely.

Fine-tuning strategy:

Base: yangheng/deberta-v3-base-absa-v1.1 while preserving the original input format [sentence] [SEP] [aspect]
CrossEntropyLoss + class weights to handle imbalance + label_smoothing=0.1 for calibration
Discriminative LR: backbone lr, classifier head lr × 5
Early stopping monitors Val F1-Macro

Confidence threshold: Calibrate on the validation set after training → automatically select CONF_THRESHOLD_M2.

**NOTE**: Due to the lack of Neutral data in labeled data, we choose to group Neutral and Positive reviews together, as it is the simpliest way as well as one of the best way to deal with this right now. Again, psuedo-labeled data can help on allow us to improve on this.

Transfer annotation multi-aspects to list (aspect, sentiment_int) pairs.

    aspect_str    : 'battery;camera;performance'
    sentiment_str : '-1;1;1'   ← aligned theo thứ tự, semicolon-separated

    Returns: [('battery', -1), ('camera', 1), ('performance', 0)]
    (Empty list if NaN or length mismatch)

In [19]:
SENTIMENT_LABELS = ['Negative', 'Positive']  # index 0 | 1 |
SENTIMENT_MAP = {-1: 0, 0: 1, 1: 1}  # sentiment int → class index


def parse_aspect_sentiment_pairs(aspect_str, sentiment_str, exclude=EXCLUDE_ASPECTS):
    if pd.isna(aspect_str) or pd.isna(sentiment_str):
        return []

    raw_aspects    = [a.strip().lower() for a in str(aspect_str).split(';')]
    raw_sentiments = [s.strip()         for s in str(sentiment_str).split(';')]

    # Drop misaligned annotation
    if len(raw_aspects) != len(raw_sentiments):
        return []

    pairs = []
    for asp, sent in zip(raw_aspects, raw_sentiments):
        if asp in exclude or not asp:
            continue
        try:
            pairs.append((asp, int(float(sent))))
        except (ValueError, TypeError):
            continue
    return pairs

In [20]:
def build_m2_triples(df_split, label_map=SENTIMENT_MAP):
    rows = []
    for _, row in df_split.iterrows():
        if row.get(CFG.IS_TECH_COL, 0) != 1 or not row.get('is_valid_pairs', False):
            continue

        sentence = str(row[CFG.TEXT_COL])
        for asp, sent_int in zip(row['aspects_parsed'], row['sentiments_parsed']):
            rows.append({
                'sentence': sentence,
                'aspect': asp,
                'label_idx': label_map.get(sent_int, 1),
            })

    return pd.DataFrame(rows)

In [21]:
print('Building M2 per-aspect triples (Neutral dropped)...')
m2_train_df = build_m2_triples(df_full_train); print(f'  → Train : {len(m2_train_df)}')
m2_val_df   = build_m2_triples(df_full_val);   print(f'  → Val   : {len(m2_val_df)}')
m2_test_df  = build_m2_triples(df_full_test);  print(f'  → Test  : {len(m2_test_df)} (LOCKED)')

print('\nTrain distribution (sau khi drop Neutral):')
dist  = m2_train_df['label_idx'].value_counts().sort_index()
total = len(m2_train_df)
for idx, cnt in dist.items():
    print(f'  {SENTIMENT_LABELS[idx]:10}: {cnt:4d} ({cnt/total*100:.1f}%)')
assert 2 not in dist.index, 'OLD LABEL IN TRAIN SET'

Building M2 per-aspect triples (Neutral dropped)...
  → Train : 911
  → Val   : 195
  → Test  : 195 (LOCKED)

Train distribution (sau khi drop Neutral):
  Negative  :  354 (38.9%)
  Positive  :  557 (61.1%)


In [22]:
class SentimentDataset(torch.utils.data.Dataset):
    """
    Input  : text=sentence, text_pair=aspect
    Label  : class index ∈ {0=Negative, 1=Positive}
    """
    def __init__(self, df, tokenizer, max_len=CFG.M2_MAX_LEN):
        # Seperate 2 list
        self.sentences = df['sentence'].tolist()
        self.aspects   = df['aspect'].tolist()

        self.labels    = df['label_idx'].tolist()
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        # Tokenizer will place [SEP] automatically
        enc = self.tokenizer(
            text=self.sentences[idx],
            text_pair=self.aspects[idx],
            truncation='only_first',
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt',
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long),
        }

In [23]:
def compute_sentiment_class_weights(train_df):
    counts = train_df['label_idx'].value_counts().sort_index()
    total = len(train_df)
    weights = []

    for i in range(2):
        weights.append(float(np.clip(total / (2 * counts.get(i, 1)), 0.2, 5.0)))

    w = torch.tensor(weights, dtype=torch.float32)
    print(f'Sentiment class weights: Neg={w[0]:.3f} | Pos_Neu={w[1]:.3f}')
    return w

In [24]:
m2_class_weights = compute_sentiment_class_weights(m2_train_df)

Sentiment class weights: Neg=1.287 | Pos_Neu=0.818


In [25]:
m2_tokenizer = AutoTokenizer.from_pretrained(CFG.SENTIMENT_MODEL)
m2_model = AutoModelForSequenceClassification.from_pretrained(
    CFG.SENTIMENT_MODEL,
    num_labels=2,
    id2label={0: 'Negative', 1: 'Positive'},
    label2id={'Negative': 0, 'Positive': 1},
    ignore_mismatched_sizes=True
)
m2_model.to(DEVICE)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/372 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/18.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `3`.


model.safetensors:   0%|          | 0.00/738M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: yangheng/deberta-v3-base-absa-v1.1
Key               | Status   |                                                                                       
------------------+----------+---------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([3, 768]) vs model:torch.Size([2, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([3]) vs model:torch.Size([2])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


DebertaV2ForSequenceClassification(
  (deberta): DebertaV2Model(
    (embeddings): DebertaV2Embeddings(
      (word_embeddings): Embedding(128100, 768, padding_idx=0)
      (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): DebertaV2Encoder(
      (layer): ModuleList(
        (0-11): 12 x DebertaV2Layer(
          (attention): DebertaV2Attention(
            (self): DisentangledSelfAttention(
              (query_proj): Linear(in_features=768, out_features=768, bias=True)
              (key_proj): Linear(in_features=768, out_features=768, bias=True)
              (value_proj): Linear(in_features=768, out_features=768, bias=True)
              (pos_dropout): Dropout(p=0.1, inplace=False)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): DebertaV2SelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): Layer

Batch inference

In [26]:
def predict_sentiment_batch_raw(sentences, aspects, tokenizer, model,
                                 batch_size=CFG.M2_INFERENCE_BATCH):
    all_probs   = []

    for i in range(0, len(sentences), batch_size):
        batch_sentences = sentences[i:i+batch_size]
        batch_aspects   = aspects[i:i+batch_size]

        enc   = tokenizer(
            text=batch_sentences,
            text_pair=batch_aspects,
            truncation='only_first',
            padding='max_length',
            max_length=CFG.M2_MAX_LEN,
            return_tensors='pt'
        ).to(DEVICE)

        with torch.no_grad():
            logits = model(**enc).logits
        probs = F.softmax(logits, dim=-1).cpu().numpy()
        all_probs.extend(probs)

    all_probs = np.array(all_probs)
    return np.argmax(all_probs, axis=-1), all_probs

Fine-tune yangheng/deberta-absa per-aspect sentiment triples.

In [27]:
def train_model2(train_df, val_df, tokenizer, model,
                 class_weights, output_dir=CFG.M2_OUTPUT,
                 epochs=CFG.M2_EPOCHS, lr=CFG.M2_LR, patience=CFG.M2_PATIENCE):
    checkpoint_path = output_dir + '/model_best_m2.pt'
    if os.path.exists(checkpoint_path) and not CFG.FORCE_RETRAIN_M2:
        print(f'M2 checkpoint found at {output_dir}, skipping training.')
        print('  → Set CFG.FORCE_RETRAIN_M2 = True to retrain.')
        return
    if os.path.exists(checkpoint_path) and CFG.FORCE_RETRAIN_M2:
        import shutil
        shutil.rmtree(output_dir)
        print(f'[FORCE_RETRAIN_M2] Delete old checkpoint {output_dir}. Train again...')

    train_ds = SentimentDataset(train_df, tokenizer)
    val_ds   = SentimentDataset(val_df,   tokenizer)

    train_loader = DataLoader(train_ds, batch_size=CFG.M2_BATCH, shuffle=True,  num_workers=2)
    val_loader   = DataLoader(val_ds,   batch_size=CFG.M2_BATCH, shuffle=False, num_workers=2)

    criterion = nn.CrossEntropyLoss(
        weight=class_weights.to(DEVICE),
        label_smoothing=0.1,
    )

    # Discriminative LR: keeping ABSA knowledge
    try:
        backbone_params   = model.deberta.parameters()
    except AttributeError:
        backbone_params   = model.base_model.parameters()
    classifier_params = model.classifier.parameters()

    optimizer = AdamW(
        [
            {'params': backbone_params,'lr': lr},
            {'params': classifier_params, 'lr': lr * 5},
        ],
        weight_decay=0.01,
    )

    total_steps = len(train_loader) * epochs
    scheduler   = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(total_steps * 0.1),
        num_training_steps=total_steps,
    )

    best_val_f1  = 0.0
    patience_ctr = 0
    os.makedirs(output_dir, exist_ok=True)

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0

        for batch in train_loader:
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels         = batch['labels'].to(DEVICE)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss    = criterion(outputs.logits, labels)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

            epoch_loss += loss.item()

        model.eval()
        val_preds_ep, val_true_ep = [], []
        val_loss_total = 0.0

        with torch.no_grad():
            for batch in val_loader:
                input_ids      = batch['input_ids'].to(DEVICE)
                attention_mask = batch['attention_mask'].to(DEVICE)
                labels         = batch['labels'].to(DEVICE)

                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                loss    = criterion(outputs.logits, labels)
                val_loss_total += loss.item()

                preds = outputs.logits.argmax(dim=-1).cpu().numpy()
                val_preds_ep.extend(preds)
                val_true_ep.extend(labels.cpu().numpy())

        val_f1  = f1_score(val_true_ep, val_preds_ep, average='macro', zero_division=0)
        val_acc = (np.array(val_preds_ep) == np.array(val_true_ep)).mean()
        n_tr    = len(train_loader)
        n_va    = len(val_loader)

        print(f'Epoch {epoch+1}/{epochs} | '
              f'Train Loss={epoch_loss/n_tr:.4f} | '
              f'Val Loss={val_loss_total/n_va:.4f} | '
              f'Val F1-Macro={val_f1:.4f} | Val Acc={val_acc:.4f}')

        if val_f1 > best_val_f1:
            best_val_f1  = val_f1
            patience_ctr = 0
            torch.save(model.state_dict(), output_dir + '/model_best_m2.pt')
            print(f'  ✓ New best Val F1-Macro={best_val_f1:.4f} — saved.')
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                print(f'  Early stopping at epoch {epoch+1}')
                break

    with open(output_dir + '/m2_config.json', 'w') as f:
        json.dump({
            'base_model':  CFG.SENTIMENT_MODEL,
            'best_val_f1': best_val_f1,
            'n_classes':   2,
            'id2label':    {0: 'Negative', 1: 'Positive'},
        }, f, indent=2)

    print(f'M2 training done. Best Val F1-Macro: {best_val_f1:.4f}')

In [28]:
def run_m2_calibration(m2_val_df, tokenizer, model, default_th=0.65):
    print('Computing M2 predictions on val set for calibration...')
    val_preds, val_probs = predict_sentiment_batch_raw(
        m2_val_df['sentence'].tolist(),
        m2_val_df['aspect'].tolist(),
        tokenizer, model,
    )
    val_true = m2_val_df['label_idx'].values
    val_conf = val_probs.max(axis=1)

    print(f'Val set: {len(val_preds)} aspect-pairs (Binary: Neg/Pos)')
    print(f'Baseline — Acc={(val_preds==val_true).mean():.4f} | '
          f'F1={f1_score(val_true, val_preds, average="macro", zero_division=0):.4f}')

    print(f'\n{classification_report(val_true, val_preds, labels=[0, 1], target_names=SENTIMENT_LABELS, zero_division=0)}')

    print('Confidence threshold calibration (Val, Neg/Pos):')
    print(f'  {"Threshold":>9} | {"Coverage%":>9} | {"F1-Macro":>8} | {"Accuracy":>8}')
    print('  ' + '-'*46)

    best_score, best_th = 0, default_th
    for th in np.arange(0.40, 0.92, 0.05):
        mask = val_conf >= th
        if mask.sum() < 10:
            break
        cov   = mask.mean() * 100
        f1_t  = f1_score(val_true[mask], val_preds[mask], average='macro', zero_division=0)
        acc_t = (val_preds[mask] == val_true[mask]).mean()
        flag  = ' ◀' if f1_t > best_score else ''
        print(f'  {th:>9.2f} | {cov:>9.1f} | {f1_t:>8.4f} | {acc_t:>8.4f}{flag}')
        if f1_t > best_score:
            best_score, best_th = f1_t, th

    print(f'\n→ CONF_THRESHOLD_M2 = {best_th:.2f}  (Val F1={best_score:.4f})')
    print('  Override manually trade off coverage / precision.')
    return best_th

In [29]:
train_model2(
    train_df=m2_train_df, val_df=m2_val_df,
    tokenizer=m2_tokenizer, model=m2_model,
    class_weights=m2_class_weights,
)

#  Load best fine-tuned checkpoint
ckpt_path = CFG.M2_OUTPUT + '/model_best_m2.pt'
if not os.path.exists(ckpt_path):
    raise FileNotFoundError(
        f'Checkpoint does not exists {ckpt_path}. '
        'Please rerun the cell train or check the CFG.M2_OUTPUT.'
    )
m2_model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
m2_model.eval()
print('M2 fine-tuned model loaded.')

CONF_THRESHOLD_M2 = run_m2_calibration(m2_val_df, m2_tokenizer, m2_model)

# Save threshold
with open(CFG.M2_OUTPUT + '/m2_threshold.json', 'w') as f:
    json.dump({'conf_threshold_m2': CONF_THRESHOLD_M2}, f, indent=2)
print(f'\nFinal CONF_THRESHOLD_M2 = {CONF_THRESHOLD_M2:.2f} saved.')

Epoch 1/10 | Train Loss=0.5051 | Val Loss=0.3764 | Val F1-Macro=0.8784 | Val Acc=0.8821
  ✓ New best Val F1-Macro=0.8784 — saved.
Epoch 2/10 | Train Loss=0.3596 | Val Loss=0.3236 | Val F1-Macro=0.9158 | Val Acc=0.9179
  ✓ New best Val F1-Macro=0.9158 — saved.
Epoch 3/10 | Train Loss=0.2933 | Val Loss=0.3537 | Val F1-Macro=0.9152 | Val Acc=0.9179
Epoch 4/10 | Train Loss=0.2579 | Val Loss=0.3847 | Val F1-Macro=0.8983 | Val Acc=0.9026
Epoch 5/10 | Train Loss=0.2458 | Val Loss=0.3491 | Val F1-Macro=0.9197 | Val Acc=0.9231
  ✓ New best Val F1-Macro=0.9197 — saved.
Epoch 6/10 | Train Loss=0.2346 | Val Loss=0.3715 | Val F1-Macro=0.9097 | Val Acc=0.9128
Epoch 7/10 | Train Loss=0.2257 | Val Loss=0.3907 | Val F1-Macro=0.9086 | Val Acc=0.9128
Epoch 8/10 | Train Loss=0.2233 | Val Loss=0.4044 | Val F1-Macro=0.8983 | Val Acc=0.9026
  Early stopping at epoch 8
M2 training done. Best Val F1-Macro: 0.9197
M2 fine-tuned model loaded.
Computing M2 predictions on val set for calibration...
Val set: 195 as

## 8. Evaluation on Global Test Set (open once)

In [30]:
print('GLOBAL TEST SET EVALUATION — MODEL 2 (Fine-tuned Sentiment)')
print('Evaluating on ALL per-aspect pairs (single + multi-aspect)\n')

eval_test_df = m2_test_df.rename(columns={'label_idx': 'true_label'}).copy()
n_tech_test  = (df_full_test[CFG.IS_TECH_COL] == 1).sum()
print(f'M2 Test pairs: {len(eval_test_df)} (từ {n_tech_test} technical rows)')

test_preds, test_probs = predict_sentiment_batch_raw(
    eval_test_df['sentence'].tolist(),
    eval_test_df['aspect'].tolist(),
    m2_tokenizer, m2_model,
)
test_true = eval_test_df['true_label'].values
test_conf = test_probs.max(axis=1)

print(classification_report(test_true, test_preds, labels=[0, 1], target_names=SENTIMENT_LABELS, zero_division=0))
print(f'F1-Macro : {f1_score(test_true, test_preds, average="macro", zero_division=0):.4f}')
print(f'Accuracy : {(test_preds == test_true).mean():.4f}')

# Confident-only subset
mask_conf = test_conf >= CONF_THRESHOLD_M2
if mask_conf.sum() > 0:
    f1_conf  = f1_score(test_true[mask_conf], test_preds[mask_conf],
                         average='macro', zero_division=0)
    acc_conf = (test_preds[mask_conf] == test_true[mask_conf]).mean()
    print(f'\nConfident subset (>={CONF_THRESHOLD_M2:.2f}): '
          f'{mask_conf.sum()}/{len(test_preds)} pairs ({mask_conf.mean()*100:.1f}%)')
    print(f'  F1-Macro : {f1_conf:.4f} | Accuracy : {acc_conf:.4f}')

# Per-aspect breakdown
eval_test_df = eval_test_df.copy()
eval_test_df['pred']    = test_preds
eval_test_df['correct'] = (test_preds == test_true)
eval_test_df['conf']    = test_conf

print(f'\nPer-aspect breakdown (test, all pairs):')
print(f'  {"Aspect":<20} | {"N":>4} | {"Acc":>6} | {"F1":>6} | {"Acc(conf)":>10}')
print('  ' + '-' * 58)

for asp in ASPECT_CLASSES:
    sub = eval_test_df[eval_test_df['aspect'] == asp]
    if len(sub) == 0:
        continue
    sub_conf = sub[sub['conf'] >= CONF_THRESHOLD_M2]
    acc_all  = sub['correct'].mean()
    f1_asp   = f1_score(sub['true_label'], sub['pred'], average='macro', zero_division=0)
    acc_c    = sub_conf['correct'].mean() if len(sub_conf) > 0 else float('nan')
    print(f'  {asp:<20} | {len(sub):>4} | {acc_all:>6.3f} | {f1_asp:>6.3f} | {acc_c:>10.3f}')

GLOBAL TEST SET EVALUATION — MODEL 2 (Fine-tuned Sentiment)
Evaluating on ALL per-aspect pairs (single + multi-aspect)

M2 Test pairs: 195 (từ 182 technical rows)
              precision    recall  f1-score   support

    Negative       0.85      0.95      0.89        75
    Positive       0.96      0.89      0.93       120

    accuracy                           0.91       195
   macro avg       0.90      0.92      0.91       195
weighted avg       0.92      0.91      0.91       195

F1-Macro : 0.9097
Accuracy : 0.9128

Confident subset (>=0.90): 174/195 pairs (89.2%)
  F1-Macro : 0.9412 | Accuracy : 0.9425

Per-aspect breakdown (test, all pairs):
  Aspect               |    N |    Acc |     F1 |  Acc(conf)
  ----------------------------------------------------------
  battery              |   12 |  0.917 |  0.916 |      0.900
  design & build       |   52 |  0.885 |  0.880 |      0.951
  display & audio      |   23 |  1.000 |  1.000 |      1.000
  input                |   15 |  0.800

In [32]:
import shutil

m2_tokenizer.save_pretrained(CFG.M2_OUTPUT)
print(f'✅ Succesfully saved M2 tokenizer at {CFG.M2_OUTPUT}')

shutil.make_archive('model2_sentiment', 'zip', CFG.M2_OUTPUT)
print('✅ zipped Model 2 to model2_sentiment.zip')

print('\n🎉 Well done! Everything is saved on the side bar!')

✅ Succesfully saved M2 tokenizer at ./model2_sentiment
✅ zipped Model 2 to model2_sentiment.zip

🎉 Well done! Everything is saved on the side bar!
